# GPBam Datasets

This notebook generates the various datasets (for essay writing, article recitation [, case recitation]). 

## Imports

In [ ]:
import sys,os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import warnings

from src import data, qa

from dotenv import load_dotenv
load_dotenv()

## Essay Writing

### Data Loading

In [ ]:
case_facts = data.load_pdfs()

In [ ]:
case_solutions = data.load_pdfs('data/02_Lösungen/')

In [ ]:
gpbam_df = pd.DataFrame({'facts': case_facts, 'solutions': case_solutions})
gpbam_df.to_json('data/gpbam.json')

In [ ]:
gpbam_df = pd.read_json('data/gpbam.json')
display(gpbam_df.head())

gpbam_df.iloc[0]['solutions']

In [ ]:
from transformers import AutoTokenizer
import pandas as pd

MODEL_NAME = "jinaai/jina-embeddings-v5-text-small-retrieval"
MAX_LEN = 32768

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Count tokens without creating a new column
facts_token_length = gpbam_df["facts"].fillna("").astype(str).apply(
    lambda x: len(tokenizer.encode(x, add_special_tokens=True))
)

# Check whether each row is under the limit without creating a new column
facts_under_limit = facts_token_length <= MAX_LEN

# Summary
print("Max facts token length:", facts_token_length.max())
print("Rows over limit:", (~facts_under_limit).sum())
print("Rows under or equal limit:", facts_under_limit.sum())

### Document Analysis

In [ ]:
import tiktoken

def count_tokens(text: str, model: str = "gpt-4o"):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

In [ ]:
token_counts = pd.DataFrame({col: gpbam_df[col].apply(count_tokens) for col in gpbam_df.columns})
token_counts = token_counts.melt().rename(columns={'value': 'tokens', 'variable': 'documents'})

In [ ]:
g = sns.displot(data=token_counts, x = 'tokens', hue='documents', log_scale=True, bins=40, 
               aspect=2, height=3)
g.fig.set_dpi(200)

for ax in g.axes.ravel():
    ax.grid()
    y_vals = ax.get_ylim()
    for n in range(1,4):
        ctx = 8192 * n - (n-1)*1024
        ax.plot([ctx, ctx], y_vals, c='k', alpha=.75, linestyle='--')

### Rubric Extraction

#### Zero Shot

In [ ]:
rubric_prompt = """Generiere ein Bewertungsschema für juristische Aufsätze basierend auf einer Aufgabenstellung 
(gekennzeichnet als Aufgabenstellung) und eine Musterlösung (gekennzeichnet als Musterlösung).
Insgesamt sollen 100 Punkte vergeben werden. Dabei wird nur berücksichtigt, ob die rechtlichen Streitpunkte korrekt abgehandelt werden.
Nutze also die Musterlösung als Referenz für die im Bewertungsschema vorgegebenen Aspekte, mit mehr Punkten für in der Musterlösung ausführlich behandelte Aspekte.

{}

Bewertungsschema:"""

rubric_sys_prompt = """You are a legal grading rubric generator for German law."""

rubric_extractor = qa.AnswerGenerator(prompt=rubric_prompt, system_prompt = rubric_sys_prompt)

In [ ]:
rubric_info = '\nAufgabenstellung:\n' + gpbam_df.facts + '\n\nMusterlösung:\n' + gpbam_df.solutions + '\n'

In [ ]:
rubrics = rubric_extractor.predict(rubric_info.values[:5])

In [ ]:
for r in rubrics:
    print(r, '\n\n\t===============\n\n')

#### One Shot

In [ ]:
# WITHHELD on the submission branch.  The three literals below held a complete
# GPBam case, its reference solution and its grading rubric, used as the one-shot
# example for rubric generation.  The prompt scaffold underneath is the method and
# is kept verbatim; the example itself is examination material and is not published.
# See SUBMISSION.md.
shot_facts = '''<withheld: one GPBam Sachverhalt>'''

shot_rubric = '''<withheld: one GPBam grading rubric>'''


shot_sample_solution = '''<withheld: one GPBam reference solution>'''

os_rubric_prompt = f"""Generiere ein Bewertungsschema für juristische Aufsätze basierend auf einer Aufgabenstellung 
(gekennzeichnet als Aufgabenstellung) und eine Musterlösung (gekennzeichnet als Musterlösung).
Insgesamt sollen 100 Punkte vergeben werden. Dabei wird nur berücksichtigt, ob die rechtlichen Streitpunkte korrekt abgehandelt werden.
Nutze also die Musterlösung als Referenz für die im Bewertungsschema vorgegebenen Aspekte, mit mehr Punkten für in der Musterlösung ausführlich behandelte Aspekte.

Orientiere dich an diesem Beispiel für ein perfektes Bewertungsschema (gekennzeichnet als Beispiel Bewertungsschema) basierend auf einem Beispiel Sachverhalt (gekennzeichnet als Beispiel Sachverhalt) und einer Beispiel Musterlösung (gekennzeichnet als Beispiel Musterlösung).

Beispiel Sachverhalt:
{shot_facts}

Beispliel Musterlösung:
{shot_sample_solution}

Beispiel Bewertungsschema:
{shot_rubric}


{{}}

Bewertungsschema:"""

rubric_sys_prompt = """You are a legal grading rubric generator for German law."""

os_rubric_extractor = qa.AnswerGenerator(prompt=os_rubric_prompt, system_prompt = rubric_sys_prompt,
                                        model = 'mistralai/mistral-large-2512')

In [ ]:
os_rubrics = os_rubric_extractor.predict(rubric_info.values)

In [ ]:
gpbam_df['rubric'] = os_rubrics
gpbam_df.to_json('data/gpbam_w_rubric.json')

## Article Recall

### Most cited laws

As given by the OLD201k graph

#### Data Loading

In [ ]:
old_ref_fn = 'data/old/old_ref.pkl'

if not os.path.exists(old_ref_fn):
    with open(old_ref_fn, 'wb') as fb:
        G_ = data.from_neo4j()
        pickle.dump(G_, fb)
    
old_g, old_cases, old_laws = data.load_old_graph(old_ref_fn)
old_g.name = 'OLD201k'

old_law_texts = old_laws.content.fillna('').values

#### Extraction

In [ ]:
most_cited_df = pd.read_csv('data/most_cited_laws.csv')
most_cited_df.rename(columns={'code':'law_book', 'section': 'article'}).to_csv('data/most_cited_laws.csv', index=False)

In [ ]:
law_in_degs = old_g.in_degrees(etype='CL')
old_laws['citations'] = law_in_degs

most_cited = law_in_degs.argsort(descending=True)[:100]

most_cited_df = old_laws.iloc[most_cited].rename(columns={'code':'law_book', 'section': 'article'})

most_cited_df.to_csv('data/most_cited_laws.csv', index=False)

In [ ]:
most_cited_df

#### Analysis

In [ ]:
old_laws.groupby('code').citations.sum().sort_values()[-20:].plot(kind='bar')

In [ ]:
most_cited_df.groupby('code').section.count().sort_values().plot(kind='bar')

### GPBam Laws

As given by the GPBam solutions. Overlap between task and GPBam solution is not considered!

#### Data Loading

In [ ]:
gpbam_df = pd.read_json('data/gpbam.json')

#### Extraction

In [ ]:
from refex.extractor import RefExtractor
from refex.errors import RefExError

extractor = RefExtractor()

def extract_refs(s):
    def get_refs(markers):
        for marker in markers:
            for ref in marker.get_references():
                if ref.book not in (None, "") and ref.section not in (None, ""):
                    yield (ref.book, ref.section)
    
    try:
        content, markers = extractor.extract(s)
        return list(get_refs(markers))
    except RefExError as e:
        return None

In [ ]:
refs = gpbam_df.solutions.apply(extract_refs)
references_df =  []

for i, refs_list in enumerate(refs):
    df = pd.DataFrame(refs_list, columns=('law_book', 'article'))
    df['index'] = i
    references_df.append(df)

references_df = pd.concat(references_df).dropna()

In [ ]:
references_df['count'] = 1
references_df = pd.DataFrame(references_df.groupby(['law_book', 'article'])['count'].count()).reset_index().sort_values('count')
top_references_df = references_df.sort_values('count', ascending=False).head(150)

In [ ]:
data.ensure_law_data()
laws = pd.DataFrame(data.parse_german_laws())

In [ ]:
def get_law(book, paragraph, aliases = {'baugb' : 'bbaug'}):
    if book in aliases:
        book = aliases[book]
    is_book = laws.law_book.str.split(' ', expand=True)[0].str.lower() == book
    is_paragraph = laws.paragraph == f"§ {paragraph}"
    if np.sum(is_book * is_paragraph) == 0:
        warnings.warn(f'Could not find {book} {paragraph}')
        return None
    return laws[is_book & is_paragraph]['text'].values[0]

top_refs_content = [get_law(b, p) for b,p in zip(top_references_df['law_book'], top_references_df['article'])]

In [ ]:
top_references_df['content'] = top_refs_content
top_references_df = top_references_df.dropna().head(100)
top_references_df.to_csv('data/gp_laws.csv', index=False)

#### Analysis

In [ ]:
references_df.reset_index().groupby('law_book').article.count().sort_values()[-20:].plot(kind='bar')

In [ ]:
top_references_df.groupby('law_book')['count'].sum().sort_values()[-20:].plot(kind='bar')

## Case Recall

### GPB Cases

#### Data Loading

In [ ]:
gpbam_df = pd.read_json('data/gpbam.json')

#### Extraction

In [ ]:
ce_prompt = """Extrahiere alle Referenzen auf Urteile von deutschen Gerichten aus diesem Gutachten. 
Die meisten dieser Referenzen sind als Fußntoten ausgeführt.
Bestimme für jedes Urteil das Gericht, Datum und Aktenzeichen und generiere den ECLI (European Case-Law Identifier) daraus.

Antworte nur mit den extrahierten Urteilen im JSON format [{{"name": "Urteil vom 12.03.2015 - BVerwG 3 C 28.13", "ECLI": "ECLI:DE:BVerwG:2015:120315U3C28.13.0", "court": "BVerwG", "date": "2015:1203", "surface_form": "1 Urteil vom 12.03.2015 BVerwG 3 C 28.13"}}, ...].
Falls es keine Urteile gibt, antworte nur mit einer leeren Liste [] ohne andere Erklärungen.

Gutachten:
{}

Referenzen:"""

ce_sys_prompt = """You are a case reference extractor for German law. 
Given a text, you find all references to cases and generate the ECLI for them.
Always return properly formatted JSON data and nothing else."""

case_extractor = qa.AnswerGenerator(prompt=ce_prompt, system_prompt = ce_sys_prompt)

In [ ]:
cases = case_extractor.predict(gpbam_df.solutions.values)

In [ ]:
def parse(c):
    try:
        return json.loads(c)
    except json.JSONDecodeError:
        return None

cases = [parse(c) for c in cases]
cases_df = pd.concat([pd.DataFrame(c) for c in cases])
cases_df['count'] = 1
cases_df = pd.DataFrame(cases_df.groupby(['ECLI', 'court', 'date', 'surface_form'])['count'].count()).reset_index()

In [ ]:
top_cases_df = cases_df.sort_values('count', ascending=False).head(100)
top_cases_df.to_csv('data/gp_cases.csv')

#### Analysis

In [ ]:
cases_df.reset_index().groupby('court').ECLI.count().sort_values()[-20:].plot(kind='bar')

In [ ]:
top_cases_df.groupby('court')['count'].sum().sort_values()[-20:].plot(kind='bar')